In [1]:
import os
import shutil

# 1. Locate the attached codebase
src_dir = None
req_file = None
for dirname, dirs, files in os.walk('/kaggle/input'):
    if 'src' in dirs and 'chakramodel' in dirname.lower():
        src_dir = os.path.join(dirname, 'src')
    if 'requirements.txt' in files:
        req_file = os.path.join(dirname, 'requirements.txt')

if src_dir and req_file:
    print(f"Found code at: {src_dir}")
    if os.path.exists('./src'): shutil.rmtree('./src')
    shutil.copytree(src_dir, './src')
    os.system(f"pip install -r {req_file} -q")
else:
    print("WARNING: Code dataset not found attached, but YOLO training will still proceed.")

# 2. Install YOLO requirements
!pip install ultralytics requests opencv-python-headless -q

Found code at: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.8 MB/s eta 0:00:00


In [2]:
import os
import cv2
import shutil
import numpy as np
import urllib.request
import zipfile
import ssl
from pathlib import Path
from ultralytics import YOLO

# Bypass SSL certificate errors for downloads
ssl._create_default_https_context = ssl._create_unverified_context

dataset_dir = Path("/kaggle/working/polyp_dataset")
images_dir = dataset_dir / "images"
labels_dir = dataset_dir / "labels"

for split in ['train', 'val']:
    (images_dir / split).mkdir(parents=True, exist_ok=True)
    (labels_dir / split).mkdir(parents=True, exist_ok=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [4]:
all_data = []

# Automatically discover all images and masks in your attached Kaggle datasets!
print("Searching attached datasets for images and masks...")
for dirname, dirs, files in os.walk('/kaggle/input'):
    dirname_lower = dirname.lower()
    # Find folders that look like they contain images
    if 'image' in dirname_lower or 'original' in dirname_lower:
        parent = Path(dirname).parent
        
        # Look for the matching mask folder next to it
        possible_mask_dirs = [
            parent / 'mask', parent / 'masks', 
            parent / 'Ground Truth', parent / 'ground_truth', 
            parent / 'GT'
        ]
        
        mask_dir = None
        for pmd in possible_mask_dirs:
            if pmd.exists():
                mask_dir = pmd
                break
        
        # If we found a matching mask folder, pair them up!
        if mask_dir:
            for img_path in Path(dirname).glob('*.*'):
                if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg', '.tif']:
                    mask_path = mask_dir / img_path.name
                    if mask_path.exists():
                        all_data.append((img_path, mask_path))

# Remove duplicates just in case
all_data = list(set(all_data))
print(f"✅ Successfully found {len(all_data)} image-mask pairs from your attached datasets!")

# Build YOLO Dataset
print("Building YOLO dataset (this may take a minute)...")
np.random.seed(42)
np.random.shuffle(all_data)
train_split = int(0.85 * len(all_data))

for i, (img_p, msk_p) in enumerate(all_data):
    split = "train" if i < train_split else "val"
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    yolo_lines = mask_to_yolo(msk_p, w, h)
    if not yolo_lines: continue
    
    new_img_name = f"{img_p.stem}_{i}.jpg"
    new_txt_name = f"{img_p.stem}_{i}.txt"
    
    cv2.imwrite(str(images_dir / split / new_img_name), img)
    with open(labels_dir / split / new_txt_name, "w") as f:
        f.write("\n".join(yolo_lines))

print("🎉 YOLO Dataset generated successfully!")

Searching attached datasets for images and masks...
✅ Successfully found 1500 image-mask pairs from your attached datasets!
Building YOLO dataset (this may take a minute)...


NameError: name 'mask_to_yolo' is not defined

In [5]:
def mask_to_yolo(mask_path, img_w, img_h):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    
    # FIX: Strict threshold to remove compression noise (prevents 12,000+ fake boxes)
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 0 and h > 0:
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            nw = w / img_w
            nh = h / img_h
            bboxes.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    return bboxes

In [6]:
yaml_content = f"""path: {dataset_dir}
train: images/train
val: images/val

names:
  0: polyp
"""
with open(dataset_dir / "dataset.yaml", "w") as f:
    f.write(yaml_content)

In [7]:
# Train across your 2x T4 GPUs
model = YOLO("yolov8n.pt") 

results = model.train(
    data=str(dataset_dir / "dataset.yaml"),
    epochs=50,
    imgsz=640,
    batch=32, 
    device=[0, 1], 
    project="/kaggle/working/yolo_output",
    name="cross_dataset_polyp"
)

print("Training Complete! Find weights at /kaggle/working/yolo_output/cross_dataset_polyp/weights/best.pt")

Ultralytics 8.4.140 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)


KeyboardInterrupt: 